# 07: RAGAS / Answer-Quality Evaluation

Phase 2. Adds answer-quality evaluation on top of the IR metrics: runs RAGAS and a
correctness check across all four retrieval configs, judged by `gpt-5.5` via the OpenAI
API — independent of both the Llama-3.1-8B generator and the Qwen2.5 reference-answer
drafters (both 14B and 72B were run and kept for comparison, see `data/test_set.jsonl`'s
`reference_14b`/`reference_72b` fields). Run after notebooks 03 (baseline) and 06 (hybrid).

See `07_ragas_evaluation.md` for the build plan and completion criteria, and
`ragas/07_ragas_evaluation.md` for full technical detail and decisions.

## Cell group 1: Imports and constants

In [3]:
import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

SCRIPTS_DIR = Path("scripts").resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from log_setup import setup_logging
logger = setup_logging("07_ragas_evaluation")

DATA_DIR    = Path("data")
CHROMA_DIR  = DATA_DIR / "chroma_db"
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

EVAL_K       = [1, 3, 5, 10]
RRF_K        = 60
GRAPH_HOPS   = 2
TOP_K        = 10

JUDGE_MODEL  = os.getenv("OPENAI_MODEL", "gpt-5.5")
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"

ALL_CONFIGS = {
    'dense_only':   'Dense only',
    'sparse_only':  'Sparse only',
    'dense_sparse': 'Dense + Sparse',
    'hybrid':       'Hybrid (+ Graph)',
}

print(f'JUDGE_MODEL: {JUDGE_MODEL}')
print(f'EMBED_MODEL: {EMBED_MODEL}')
print(f'Configs: {list(ALL_CONFIGS)}')

16:36:04  INFO      === Notebook 07_ragas_evaluation started, log: 2026-07-06_16-36-04_07_ragas_evaluation.log ===


JUDGE_MODEL: gpt-5.5
EMBED_MODEL: sentence-transformers/all-MiniLM-L6-v2
Configs: ['dense_only', 'sparse_only', 'dense_sparse', 'hybrid']


## Quick check: is the OpenAI API reachable?

Minimal connectivity check before building anything on top of it — same idea as
`verify_api_key.py`, condensed into one cell.

In [4]:
from openai import OpenAI, APIError

client = OpenAI()
t0 = time.time()
try:
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
    )
    elapsed = time.time() - t0
    reply = resp.choices[0].message.content.strip()
    print(f'OK  {JUDGE_MODEL}  ({elapsed:.2f}s)  reply={reply!r}')
except APIError as e:
    elapsed = time.time() - t0
    print(f'FAILED  ({elapsed:.2f}s)  {type(e).__name__}: {e}')
    raise

16:36:39  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


OK  gpt-5.5  (3.43s)  reply='OK'


## Cell group 2: Judge and embeddings setup

`ragas==0.4.3` hard-imports a `langchain_community` submodule that no longer exists in
the installed package (confirmed: current latest release, not a version we chose badly —
see `ragas/07_ragas_evaluation.md`). The stub below must run before the first `import
ragas` anywhere in this notebook.

In [5]:
import types

if "langchain_community.chat_models.vertexai" not in sys.modules:
    _stub = types.ModuleType("langchain_community.chat_models.vertexai")
    _stub.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules["langchain_community.chat_models.vertexai"] = _stub

import ragas
print(f'ragas {ragas.__version__} imported OK')

C:\Users\tsono\Documents\uoe\disertation\plan\implementation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ragas 0.4.3 imported OK


In [6]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.huggingface_provider import HuggingFaceEmbeddings as RagasHFEmbeddings

# Async client: ragas's collections metrics (Faithfulness, etc.) use ascore()/agenerate()
# internally, which requires an async client. A sync client fails at call time with
# "Cannot use agenerate() with a synchronous client." Jupyter also runs its own event
# loop, so the *sync* score()/generate() path doesn't work here either way - ascore()
# with an async client is the only combination that works inside a notebook cell.
async_client = AsyncOpenAI()
judge = llm_factory(JUDGE_MODEL, client=async_client)

# Second ragas 0.4.3 bug, distinct from the vertexai one: InstructorLLM's reasoning-model
# detection (ragas/llms/base.py: _map_openai_params/is_reasoning_model) does
# int(version_str) on the bit after "gpt-" to decide if a model needs max_completion_tokens
# instead of max_tokens (plus temperature forced to 1.0, top_p removed - all three are
# gated on the same detection). int("5.5") raises ValueError, so "gpt-5.5" silently fails
# the check and is treated as a legacy model - the API then rejects the resulting request
# for using max_tokens (confirmed live, same error verify_api_key.py hit earlier).
# Patch model_args directly to what the correct branch would have produced.
# max_completion_tokens raised from ragas's own default (1024) to 4096: verified live
# against the full 200-row test set that 1024 is too tight for context_recall's
# claim-by-claim breakdown on some rows - IncompleteOutputException on at least one row
# in a 20-row sample, which (since abatch_score's asyncio.gather has no per-row
# exception isolation) silently loses the whole batch's results, not just that row.
judge.model_args["max_completion_tokens"] = 4096
judge.model_args["temperature"] = 1.0
judge.model_args.pop("top_p", None)
judge.model_args.pop("max_tokens", None)

print(f'judge OK: {type(judge).__name__}  model={JUDGE_MODEL}  is_async={judge.is_async}')
print(f'model_args (patched): {judge.model_args}')

embeddings = RagasHFEmbeddings(model=EMBED_MODEL, use_api=False)
print(f'embeddings OK: {type(embeddings).__name__}  model={EMBED_MODEL}')

judge OK: InstructorLLM  model=gpt-5.5  is_async=True
model_args (patched): {'temperature': 1.0, 'max_completion_tokens': 4096}


16:37:59  INFO      No device provided, using cuda:0


16:37:59  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


16:38:00  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


16:38:00  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


...

16:38:02  INFO      HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"


16:38:02  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"


16:38:02  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


16:38:02  INFO      HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


embeddings OK: HuggingFaceEmbeddings  model=sentence-transformers/all-MiniLM-L6-v2


In [7]:
from ragas.metrics.collections import Faithfulness, ContextRecall, ContextPrecisionWithReference, AnswerRelevancy

metric_faithfulness = Faithfulness(llm=judge)
metric_context_recall = ContextRecall(llm=judge)
metric_context_precision = ContextPrecisionWithReference(llm=judge)
metric_answer_relevancy = AnswerRelevancy(llm=judge, embeddings=embeddings)

print('All four metric objects constructed OK:')
for m in [metric_faithfulness, metric_context_recall, metric_context_precision, metric_answer_relevancy]:
    print(f'  {m.name}')

All four metric objects constructed OK:
  faithfulness
  context_recall
  context_precision_with_reference
  answer_relevancy


In [8]:
# Tiny synthetic smoke test - not real project data, just confirming the call shape works end to end.
result = await metric_faithfulness.ascore(
    user_input="Where was Einstein born?",
    response="Einstein was born in Germany.",
    retrieved_contexts=["Albert Einstein was born in Ulm, in the Kingdom of Wurttemberg in the German Empire, on 14 March 1879."],
)
print(f'type: {type(result).__name__}')
print(f'value: {result.value}')

16:38:32  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


16:38:40  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


type: MetricResult
value: 1.0


In [9]:
# Confirm the bulk path too - this is what cell groups 4-5 will actually use for 50-200 rows.
results = await metric_faithfulness.abatch_score([
    dict(
        user_input="Where was Einstein born?",
        response="Einstein was born in Germany.",
        retrieved_contexts=["Albert Einstein was born in Ulm, in the Kingdom of Wurttemberg in the German Empire, on 14 March 1879."],
    ),
    dict(
        user_input="Where was Einstein born?",
        response="Einstein was born in France.",
        retrieved_contexts=["Albert Einstein was born in Ulm, in the Kingdom of Wurttemberg in the German Empire, on 14 March 1879."],
    ),
])
for r in results:
    print(f'{r.value}')

16:38:56  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


16:38:58  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


16:39:00  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


16:39:00  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


1.0
0.0


## Cell group 3: Load retrievers and test set

Loads the same dense+sparse+graph retrievers as notebooks 05/06 (`load_retrievers()`),
plus `data/test_set.jsonl`, which now carries **both** `reference_14b` and `reference_72b`
(not a single `reference` field — see the 2026-07-05 Change log entry in
`07_ragas_evaluation.md`, since both drafter models are being kept for comparison).

In [10]:
import time
from retriever import load_retrievers, dense_only_retrieve, sparse_only_retrieve, dense_sparse_retrieve, hybrid_retrieve

t0 = time.time()
vectorstore, bm25, clauses, G = load_retrievers(DATA_DIR, CHROMA_DIR)
clause_lookup = {c['clause_id']: c for c in clauses}
print(f'Loaded {len(clauses)} clauses  ({time.time()-t0:.1f}s)')
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

TEST_SET_PATH = DATA_DIR / 'test_set.jsonl'
test_set = [
    json.loads(line)
    for line in TEST_SET_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
print(f'\nLoaded {len(test_set)} test queries')
missing_ref = [q for q in test_set if not q.get('reference_14b') or not q.get('reference_72b')]
print(f'Rows missing reference_14b or reference_72b: {len(missing_ref)}')
print(f'Example row keys: {list(test_set[0].keys())}')

16:39:40  INFO      Loading all retrievers (dense+sparse+graph) from data


16:39:40  INFO      Loading dense+sparse retrievers from data


16:39:40  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


16:39:40  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"


16:39:41  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"


...

16:39:43  INFO      HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


16:39:44  INFO      Loaded 2568 clauses


16:39:45  INFO      BM25 index built


16:39:45  INFO      NetworkX graph built: 2826 nodes, 5299 edges


Loaded 2568 clauses  (4.5s)
Graph: 2826 nodes, 5299 edges

Loaded 50 test queries
Rows missing reference_14b or reference_72b: 0
Example row keys: ['query', 'gold_ids', 'query_type', 'reference_14b', 'reference_72b']


## Cell group 4: Rebuild top-k contexts for all four configs

Same four retrieval functions as notebook 06's ablation (`dense_only_retrieve`,
`sparse_only_retrieve`, `dense_sparse_retrieve`, `hybrid_retrieve`), run here over all 50
queries so each config's retrieved contexts are available for both answer generation and
retrieval-side RAGAS. All local, no GPU/API cost.

In [12]:
def retrieve_for_config(config: str, query: str) -> list:
    if config == 'dense_only':
        return dense_only_retrieve(vectorstore, clauses, query, k=TOP_K)
    if config == 'sparse_only':
        return sparse_only_retrieve(bm25, clauses, query, k=TOP_K)
    if config == 'dense_sparse':
        return dense_sparse_retrieve(query, vectorstore, bm25, clauses, k=TOP_K, rrf_k=RRF_K)
    if config == 'hybrid':
        return hybrid_retrieve(query, vectorstore, bm25, clauses, G, k=TOP_K, graph_hops=GRAPH_HOPS, rrf_k=RRF_K)
    raise ValueError(f'unknown config: {config}')

t0 = time.time()
retrieval_results = {}
for config in ALL_CONFIGS:
    rows = []
    for item in test_set:
        retrieved = retrieve_for_config(config, item['query'])
        rows.append({**item, 'retrieved': retrieved})
    retrieval_results[config] = rows
    n_empty = sum(1 for r in rows if not r['retrieved'])
    print(f'{config:<14} {len(rows)} queries retrieved  (top-1 empty: {n_empty})')

print(f'\nTotal retrieval time: {time.time()-t0:.1f}s')

dense_only     50 queries retrieved  (top-1 empty: 0)


sparse_only    50 queries retrieved  (top-1 empty: 0)


16:42:47  INFO      dense_sparse_retrieve: query='Under MLR 2017 regulation 4, at what point is an estate agen'  top-1=mlr_2017_reg_4


16:42:48  INFO      dense_sparse_retrieve: query='What ownership or voting rights threshold must an individual'  top-1=mlr_2017_reg_5


16:42:49  INFO      dense_sparse_retrieve: query='Under MLR 2017 regulation 6, which categories of persons are'  top-1=jmlsg_1_5.3.260


...

16:43:29  INFO      hybrid_retrieve: query='MLR 2017 regulation 33 requires enhanced CDD where risks ari'  graph_expanded=10  top-1=jmlsg_1_4.10


16:43:29  INFO      hybrid_retrieve: query='MLR 2017 regulation 44 requires trustees to maintain records'  graph_expanded=16  top-1=jmlsg_1_8.24


16:43:29  INFO      hybrid_retrieve: query='Section 332 of POCA 2002 creates a failure-to-disclose offen'  graph_expanded=15  top-1=poca_2002_s333A


16:43:30  INFO      hybrid_retrieve: query="MLR 2017 regulation 3 defines 'nominated officer' as a perso"  graph_expanded=3  top-1=jmlsg_1_3.20


hybrid         50 queries retrieved  (top-1 empty: 0)

Total retrieval time: 48.6s


## Cell group 5: Load generated answers

Generation runs separately, on an H200 cluster (BF16, the same fixed
`Llama-3.1-8B-Instruct` generator used everywhere else in the project — see `plan.md`
Sec 5, "Generator model identity"). A local 6GB GPU would need ~7+ hours for all 200
generations (4 configs x 50 queries), so it isn't run here; the cluster script writes
`results/answers.jsonl`, which is loaded directly below.

In [13]:
ANSWERS_PATH = RESULTS_DIR / 'answers.jsonl'

if ANSWERS_PATH.exists():
    answers = [
        json.loads(line)
        for line in ANSWERS_PATH.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]
    logger.info('Loaded %d generated answers from %s', len(answers), ANSWERS_PATH)
    print(f'Loaded {len(answers)} answers  (expected {len(ALL_CONFIGS) * len(test_set)})')
    by_config = {c: sum(1 for a in answers if a['config'] == c) for c in ALL_CONFIGS}
    print(f'Per config: {by_config}')
    n_errors = sum(1 for a in answers if a['answer'].startswith('Generation error:'))
    print(f'Generation errors: {n_errors}')
else:
    answers = []
    print(f'{ANSWERS_PATH} not found yet.')
    print('Run the generation script on the cluster, then copy')
    print(f'results/answers.jsonl back to {RESULTS_DIR.resolve()} before continuing.')

16:43:44  INFO      Loaded 200 generated answers from results\answers.jsonl


Loaded 200 answers  (expected 200)
Per config: {'dense_only': 50, 'sparse_only': 50, 'dense_sparse': 50, 'hybrid': 50}
Generation errors: 3


## Cell group 6: Retrieval-side RAGAS (context_recall + context_precision)

Scores every (config, query, reference drafter) combination: 4 configs x 50 queries x 2
reference sets (`reference_14b`/`reference_72b`) = 400 rows, each scored on both
`context_recall` and `context_precision`. Contexts are the **full** clause texts from
cell group 4 — not the 800-char truncation the generator prompt used, since that
truncation existed only for the 8B model's context window and would distort retrieval
metrics here.

Run in chunks of 5 rows: `abatch_score()` is a bare `asyncio.gather()` with no per-row
exception isolation (verified from source), so one failing row loses its whole chunk —
small chunks cap the blast radius. Every chunk is appended to
`results/ragas_retrieval_scores.jsonl` as soon as it completes, and already-scored rows
are skipped on re-run, so a failure partway through never costs completed work. Delete
the file for a clean rerun.

In [14]:
# Live shape/timing test on 2 real rows before committing to the full 400-row run:
# real retrieved clause text, real reference answers, both metrics. Gives an actual
# per-row cost to extrapolate from instead of assuming.
_test_rows = retrieval_results['hybrid'][:2]
_test_inputs = [
    dict(
        user_input=row['query'],
        retrieved_contexts=[c['text'] for c in row['retrieved']],
        reference=row['reference_72b'],
    )
    for row in _test_rows
]

t0 = time.time()
_recall = await metric_context_recall.abatch_score(_test_inputs)
t_recall = time.time() - t0

t0 = time.time()
_precision = await metric_context_precision.abatch_score(_test_inputs)
t_precision = time.time() - t0

for row, rec, prec in zip(_test_rows, _recall, _precision):
    print(f'{row["query"][:60]!r}')
    print(f'  context_recall={rec.value:.3f}  context_precision={prec.value:.3f}')
print(f'\nrecall batch: {t_recall:.1f}s, precision batch: {t_precision:.1f}s for {len(_test_inputs)} rows')
n_total = len(ALL_CONFIGS) * 2 * len(test_set)
est_min = (t_recall + t_precision) / len(_test_inputs) * n_total / 60
print(f'Naive estimate for all {n_total} rows: ~{est_min:.0f} min (chunks of 5 run concurrently, so likely less)')

18:40:29  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:40:37  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:40:40  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:40:40  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:40:44  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

18:41:10  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:41:10  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:41:14  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:41:14  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


'Under MLR 2017 regulation 4, at what point is an estate agen'
  context_recall=1.000  context_precision=1.000
'What ownership or voting rights threshold must an individual'
  context_recall=1.000  context_precision=0.747

recall batch: 11.7s, precision batch: 37.1s for 2 rows
Naive estimate for all 400 rows: ~163 min (chunks of 5 run concurrently, so likely less)


In [ ]:
RETRIEVAL_SCORES_PATH = RESULTS_DIR / 'ragas_retrieval_scores.jsonl'
CHUNK_SIZE = 5
REF_MODELS = ['14b', '72b']

done = set()
if RETRIEVAL_SCORES_PATH.exists():
    for line in RETRIEVAL_SCORES_PATH.read_text(encoding='utf-8').splitlines():
        if line.strip():
            r = json.loads(line)
            done.add((r['config'], r['ref_model'], r['query_idx']))
n_total = len(ALL_CONFIGS) * len(REF_MODELS) * len(test_set)
print(f'Already scored: {len(done)}/{n_total} rows (resume supported)')

t0 = time.time()
n_written = 0
n_failed_chunks = 0
with RETRIEVAL_SCORES_PATH.open('a', encoding='utf-8') as f:
    for config in ALL_CONFIGS:
        for ref_model in REF_MODELS:
            rows = retrieval_results[config]
            pending = [
                (idx, row) for idx, row in enumerate(rows)
                if (config, ref_model, idx) not in done
            ]
            for start in range(0, len(pending), CHUNK_SIZE):
                chunk = pending[start:start + CHUNK_SIZE]
                inputs = [
                    dict(
                        user_input=row['query'],
                        retrieved_contexts=[c['text'] for c in row['retrieved']],
                        reference=row[f'reference_{ref_model}'],
                    )
                    for _, row in chunk
                ]
                try:
                    recall_results = await metric_context_recall.abatch_score(inputs)
                    precision_results = await metric_context_precision.abatch_score(inputs)
                except Exception as exc:
                    n_failed_chunks += 1
                    logger.warning(
                        'chunk failed (config=%s ref=%s query_idx %d-%d): %s -- rerun this cell to retry',
                        config, ref_model, chunk[0][0], chunk[-1][0], exc,
                    )
                    continue
                for (idx, row), rec, prec in zip(chunk, recall_results, precision_results):
                    f.write(json.dumps({
                        'config': config,
                        'ref_model': ref_model,
                        'query_idx': idx,
                        'query': row['query'],
                        'query_type': row.get('query_type', 'unknown'),
                        'gold_ids': row['gold_ids'],
                        'context_recall': rec.value,
                        'context_precision': prec.value,
                    }, ensure_ascii=False) + '\n')
                    n_written += 1
                f.flush()
                logger.info(
                    '%s/%s: %d/%d chunks done  (%.0fs elapsed, %d rows written)',
                    config, ref_model, start // CHUNK_SIZE + 1,
                    -(-len(pending) // CHUNK_SIZE), time.time() - t0, n_written,
                )

print(f'Done: {n_written} new rows in {(time.time()-t0)/60:.1f} min  ({n_failed_chunks} failed chunks)')
print(f'Total in file: {len(done) + n_written}/{n_total}')

Already scored: 0/400 rows (resume supported)


18:41:57  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:41:58  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:41:59  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:41:59  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

18:42:41  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:42:43  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:42:45  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:42:45  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


18:42:47  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


## Cell group 7: Answer-side RAGAS (faithfulness + answer_relevancy)

Scores the 200 generated answers. Neither metric uses the reference answers, so this is
one score per (config, query) — 200 rows, not 400.

Two decisions:

- **Contexts are rebuilt exactly as the generator saw them**: `answers.jsonl` stores
  retrieved clause IDs, and the cluster generation script truncated each clause to 800
  chars in the prompt. Faithfulness asks whether the generator stayed grounded in *its
  input*, so the judge gets the same `[clause_id]` + 800-char-truncated text — judging
  against full clause text the generator never saw would mis-credit claims that happen
  to match text beyond the truncation point.
- **The 3 generation-error rows are written with null scores** and a
  `generation_error` flag rather than skipped, keeping the file complete at 200 rows so
  the merge step handles them explicitly instead of discovering missing rows.

Same run pattern as cell group 6: chunks of 5, incremental append to
`results/ragas_answer_scores.jsonl`, resume on re-run.

In [16]:
# Prepare the 200 answer rows for scoring: rebuild each row's contexts from its stored
# clause IDs, truncated to 800 chars exactly as the generation script formatted them
# for the generator prompt.
def generator_view_contexts(clause_ids: list[str]) -> list[str]:
    return [f"[{cid}]\n{clause_lookup[cid]['text'][:800]}" for cid in clause_ids]

answer_rows = []
for config in ALL_CONFIGS:
    config_answers = [a for a in answers if a['config'] == config]
    assert len(config_answers) == len(test_set), f'{config}: {len(config_answers)} answers'
    for idx, a in enumerate(config_answers):
        assert a['query'] == test_set[idx]['query'], f'{config}[{idx}]: query order mismatch'
        answer_rows.append({
            'config': config,
            'query_idx': idx,
            'query': a['query'],
            'query_type': a['query_type'],
            'answer': a['answer'],
            'contexts': generator_view_contexts(a['retrieved']),
            'generation_error': a['answer'].startswith('Generation error:'),
        })

n_err = sum(r['generation_error'] for r in answer_rows)
print(f'{len(answer_rows)} answer rows prepared ({n_err} generation errors, scored as null)')

# Live 2-row test on real data before the full run, same as cell group 6.
_test = [r for r in answer_rows if not r['generation_error']][:2]
_inputs_f = [dict(user_input=r['query'], response=r['answer'], retrieved_contexts=r['contexts']) for r in _test]
_inputs_r = [dict(user_input=r['query'], response=r['answer']) for r in _test]

t0 = time.time()
_faith = await metric_faithfulness.abatch_score(_inputs_f)
t_faith = time.time() - t0
t0 = time.time()
_relev = await metric_answer_relevancy.abatch_score(_inputs_r)
t_relev = time.time() - t0

for r, fv, rv in zip(_test, _faith, _relev):
    print(f'{r["config"]}[{r["query_idx"]}] {r["query"][:55]!r}')
    print(f'  faithfulness={fv.value:.3f}  answer_relevancy={rv.value:.3f}')
print(f'\nfaithfulness batch: {t_faith:.1f}s, relevancy batch: {t_relev:.1f}s for {len(_test)} rows')
est_min = (t_faith + t_relev) / len(_test) * len(answer_rows) / 60
print(f'Naive estimate for all {len(answer_rows)} rows: ~{est_min:.0f} min')

200 answer rows prepared (3 generation errors, scored as null)


21:00:02  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:00:02  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:00:13  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:00:15  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

Batches: 100%|██████████| 1/1 [00:00<00:00,  2.70it/s]


Batches: 100%|██████████| 1/1 [00:00<00:00, 26.21it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 10.00it/s]

dense_only[0] 'Under MLR 2017 regulation 4, at what point is an estate'
  faithfulness=1.000  answer_relevancy=0.882
dense_only[1] 'What ownership or voting rights threshold must an indiv'
  faithfulness=0.000  answer_relevancy=0.786

faithfulness batch: 16.6s, relevancy batch: 8.7s for 2 rows
Naive estimate for all 200 rows: ~42 min


In [ ]:
ANSWER_SCORES_PATH = RESULTS_DIR / 'ragas_answer_scores.jsonl'

done = set()
if ANSWER_SCORES_PATH.exists():
    for line in ANSWER_SCORES_PATH.read_text(encoding='utf-8').splitlines():
        if line.strip():
            r = json.loads(line)
            done.add((r['config'], r['query_idx']))
print(f'Already scored: {len(done)}/{len(answer_rows)} rows (resume supported)')

t0 = time.time()
n_written = 0
n_failed_chunks = 0
with ANSWER_SCORES_PATH.open('a', encoding='utf-8') as f:

    def write_row(row, faith, relev):
        global n_written
        f.write(json.dumps({
            'config': row['config'],
            'query_idx': row['query_idx'],
            'query': row['query'],
            'query_type': row['query_type'],
            'generation_error': row['generation_error'],
            'faithfulness': faith,
            'answer_relevancy': relev,
        }, ensure_ascii=False) + '\n')
        n_written += 1

    # Generation-error rows: null scores, written once, never sent to the judge.
    for row in answer_rows:
        if row['generation_error'] and (row['config'], row['query_idx']) not in done:
            write_row(row, None, None)
    f.flush()

    pending = [
        r for r in answer_rows
        if not r['generation_error'] and (r['config'], r['query_idx']) not in done
    ]
    for start in range(0, len(pending), CHUNK_SIZE):
        chunk = pending[start:start + CHUNK_SIZE]
        inputs_f = [dict(user_input=r['query'], response=r['answer'], retrieved_contexts=r['contexts']) for r in chunk]
        inputs_r = [dict(user_input=r['query'], response=r['answer']) for r in chunk]
        try:
            faith_results = await metric_faithfulness.abatch_score(inputs_f)
            relev_results = await metric_answer_relevancy.abatch_score(inputs_r)
        except Exception as exc:
            n_failed_chunks += 1
            logger.warning(
                'chunk failed (%s[%d] onward): %s -- rerun this cell to retry',
                chunk[0]['config'], chunk[0]['query_idx'], exc,
            )
            continue
        for row, fv, rv in zip(chunk, faith_results, relev_results):
            write_row(row, fv.value, rv.value)
        f.flush()
        logger.info(
            'answer-side: %d/%d chunks done  (%.0fs elapsed, %d rows written)',
            start // CHUNK_SIZE + 1, -(-len(pending) // CHUNK_SIZE), time.time() - t0, n_written,
        )

print(f'Done: {n_written} new rows in {(time.time()-t0)/60:.1f} min  ({n_failed_chunks} failed chunks)')
print(f'Total in file: {len(done) + n_written}/{len(answer_rows)}')

Already scored: 0/200 rows (resume supported)


21:00:47  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:00:47  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:00:48  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:00:49  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

21:01:11  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:01:14  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:01:16  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:01:18  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:01:19  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


## Cell group 8: Correctness (embedding similarity + judge grade)

Custom correctness check against the reference answers — deliberately **not** ragas's
`AnswerCorrectness`, which blends statement-level F1 and similarity into one opaque
number. Two plain, separately-reported numbers per (config, query, reference drafter):

- `cosine_sim`: sentence-transformers cosine similarity (all-MiniLM-L6-v2, the same
  model used everywhere else in the project) between the generated answer and the
  reference. Local and free.
- `llm_grade`: a strict yes/no from the `gpt-5.5` judge — does the candidate answer
  state the same substantive facts and conclusions as the reference? Stored as 1/0,
  with the raw reply kept for auditing.

Both references are used (400 rows = 4 configs x 50 queries x 2 drafters), same
chunk-of-5 + incremental-append + resume pattern, output
`results/correctness_scores.jsonl`. Generation-error rows are written with null scores
as in cell group 7.

In [18]:
import asyncio
import numpy as np

# The SentenceTransformer instance ragas built inside RagasHFEmbeddings (cell group 2) -
# reused directly for the cosine check rather than loading the same model a second time.
st_model = embeddings.model_instance

def cosine_sim(a: str, b: str) -> float:
    va, vb = st_model.encode([a, b], normalize_embeddings=True)
    return float(va @ vb)

GRADE_SYSTEM = (
    'You are grading a candidate answer against a trusted reference answer for a question '
    'about UK anti-money-laundering regulation. Reply with exactly one word: "yes" if the '
    'candidate states the same substantive facts and conclusions as the reference '
    '(different wording, ordering, or citation style is fine), otherwise "no". A candidate '
    'that misses a substantive element of the reference, adds a materially wrong claim, or '
    'answers a different question is "no".'
)

async def grade_one(query: str, reference: str, candidate: str) -> tuple[int, str]:
    resp = await async_client.chat.completions.create(
        model=JUDGE_MODEL,
        temperature=1.0,
        max_completion_tokens=2048,  # gpt-5.5 spends completion tokens on reasoning before the one-word reply
        messages=[
            {'role': 'system', 'content': GRADE_SYSTEM},
            {'role': 'user', 'content': f'Question: {query}\n\nReference answer:\n{reference}\n\nCandidate answer:\n{candidate}'},
        ],
    )
    raw = (resp.choices[0].message.content or '').strip().lower()
    return (1 if raw.startswith('yes') else 0), raw

# Live 2-row test: same answer graded against both drafters' references.
_r = next(r for r in answer_rows if not r['generation_error'])
for ref_model in ['14b', '72b']:
    ref = test_set[_r['query_idx']][f'reference_{ref_model}']
    t0 = time.time()
    grade, raw = await grade_one(_r['query'], ref, _r['answer'])
    sim = cosine_sim(_r['answer'], ref)
    print(f"{_r['config']}[{_r['query_idx']}] vs {ref_model}:  llm_grade={grade} ({raw!r})  cosine_sim={sim:.3f}  ({time.time()-t0:.1f}s)")

21:29:34  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 13.63it/s]

dense_only[0] vs 14b:  llm_grade=0 ('no')  cosine_sim=0.860  (1.4s)


21:29:35  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.67it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  7.55it/s]

dense_only[0] vs 72b:  llm_grade=0 ('no')  cosine_sim=0.947  (1.5s)


In [ ]:
CORRECTNESS_PATH = RESULTS_DIR / 'correctness_scores.jsonl'

done = set()
if CORRECTNESS_PATH.exists():
    for line in CORRECTNESS_PATH.read_text(encoding='utf-8').splitlines():
        if line.strip():
            r = json.loads(line)
            done.add((r['config'], r['ref_model'], r['query_idx']))
n_total = len(answer_rows) * len(REF_MODELS)
print(f'Already scored: {len(done)}/{n_total} rows (resume supported)')

t0 = time.time()
n_written = 0
n_failed_chunks = 0
with CORRECTNESS_PATH.open('a', encoding='utf-8') as f:

    def write_corr_row(row, ref_model, sim, grade, raw):
        global n_written
        f.write(json.dumps({
            'config': row['config'],
            'ref_model': ref_model,
            'query_idx': row['query_idx'],
            'query': row['query'],
            'query_type': row['query_type'],
            'generation_error': row['generation_error'],
            'cosine_sim': sim,
            'llm_grade': grade,
            'judge_reply': raw,
        }, ensure_ascii=False) + '\n')
        n_written += 1

    # Generation-error rows: null scores, never sent to the judge.
    for row in answer_rows:
        if row['generation_error']:
            for ref_model in REF_MODELS:
                if (row['config'], ref_model, row['query_idx']) not in done:
                    write_corr_row(row, ref_model, None, None, None)
    f.flush()

    pending = [
        (row, ref_model)
        for row in answer_rows if not row['generation_error']
        for ref_model in REF_MODELS
        if (row['config'], ref_model, row['query_idx']) not in done
    ]
    for start in range(0, len(pending), CHUNK_SIZE):
        chunk = pending[start:start + CHUNK_SIZE]
        refs = [test_set[row['query_idx']][f'reference_{ref_model}'] for row, ref_model in chunk]
        try:
            grades = await asyncio.gather(*[
                grade_one(row['query'], ref, row['answer'])
                for (row, _), ref in zip(chunk, refs)
            ])
        except Exception as exc:
            n_failed_chunks += 1
            logger.warning(
                'chunk failed (%s/%s[%d] onward): %s -- rerun this cell to retry',
                chunk[0][0]['config'], chunk[0][1], chunk[0][0]['query_idx'], exc,
            )
            continue
        for (row, ref_model), ref, (grade, raw) in zip(chunk, refs, grades):
            write_corr_row(row, ref_model, cosine_sim(row['answer'], ref), grade, raw)
        f.flush()
        logger.info(
            'correctness: %d/%d chunks done  (%.0fs elapsed, %d rows written)',
            start // CHUNK_SIZE + 1, -(-len(pending) // CHUNK_SIZE), time.time() - t0, n_written,
        )

print(f'Done: {n_written} new rows in {(time.time()-t0)/60:.1f} min  ({n_failed_chunks} failed chunks)')
print(f'Total in file: {len(done) + n_written}/{n_total}')

Already scored: 0/400 rows (resume supported)


21:29:54  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:29:54  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:29:55  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


21:29:55  INFO      HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 33.33it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 27.74it/s]

21:30:46  INFO      correctness: 10/79 chunks done  (53s elapsed, 56 rows written)
